# Deep Dive into Apache Spark Cluster Architecture

**Module**: 01_spark_fundamentals  
**Notebook**: 02_spark_architecture.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The deep internal architecture of Spark: Driver, Cluster Manager, Worker nodes, and Executors.
- **Why does this topic exist?**: To write scalable distributed code, you must understand where your code runs: on the driver thread or distributed across executor threads.
- **What problem does it solve?**: Developers often write driver-side bottlenecks (e.g. collecting massive datasets, serializing non-serializable objects) without realizing it.
- **Where is it used in real-world Data Engineering?**: Diagnosing production cluster crashes, OutOfMemory errors, and cluster sizing require architectural mastery.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Introduction to Spark |
| **Required depth** | Deep working knowledge: can draw the architecture and diagnose resource/registration issues |
| **Data Engineer** | Explain Driver, Master, Worker, and Executor responsibilities and read the Master/Driver UIs. |
| **Senior Data Engineer** | Diagnose allocation, networking, and locality problems in a running application. |
| **Spark specialist** | Reason about RPC, heartbeats, executor lifecycle, and cluster-manager behavior. |
| **Next topic** | `01_spark_fundamentals/03_spark_session_and_spark_context.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A Spark application has two different control planes: the cluster manager allocates resources, and the driver
schedules tasks. Confusing the Master with the Driver, or the Worker with the Executor, leads to wrong fixes:
adding workers does not help if the driver cannot be reached, and adding driver memory does not fix a lack of cores.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Application** | One user program and its SparkSession | Driver plus a set of executors |
| **Driver** | The orchestrator | Runs the main method, builds plans, schedules tasks, hosts the UI |
| **Master** | The resource broker | Tracks workers and allocates executors in Standalone mode |
| **Worker** | A node agent | Owns capacity and launches executor JVMs |
| **Executor** | A task runner | Runs tasks for one application and stores blocks |
| **Task** | The smallest unit of work | Processes one partition in one stage |

### How it works, step by step

1. The driver starts and opens an RPC endpoint.
2. It registers the application with the cluster manager.
3. The Master selects workers with free cores and memory and sends launch commands.
4. Workers start executor JVMs; executors register with the driver.
5. The driver sends serialized tasks to executors and tracks progress.
6. Executors fetch shuffle blocks directly from other executors; the Master is not in the data path.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Detailed Architecture Walkthrough

```
+-----------------------------------------------------------------------------------+
| DRIVER (Jupyter Container)                                                        |
|  - SparkContext & SparkSession                                                    |
|  - DAGScheduler (Creates Stages)                                                  |
|  - TaskScheduler (Dispatches Tasks)                                               |
|  - Web UI (Port 4040)                                                             |
+-------------------------+---------------------------------------------------------+
                          |
                          v RPC (spark://spark-master:7077)
+-----------------------------------------------------------------------------------+
| CLUSTER MANAGER (Standalone Master)                                               |
|  - Tracks Worker capacity (Cores & RAM)                                           |
|  - Allocates Executor slots to applications                                       |
+-------------------------+---------------------------------------------------------+
                          | Launches Executors
             +------------+------------+
             |                         |
             v                         v
+------------------------+  +------------------------+
| WORKER 1 (Port 8081)   |  | WORKER 2 (Port 8082)   |
|  Worker Daemon         |  |  Worker Daemon         |
|  +------------------+  |  |  +------------------+  |
|  | Executor JVM     |  |  |  | Executor JVM     |  |
|  | - 2 Cores        |  |  |  | - 2 Cores        |  |
|  | - 4 GB RAM       |  |  |  | - 4 GB RAM       |  |
|  | - BlockManager   |  |  |  | - BlockManager   |  |
|  | - Task Threads   |  |  |  | - Task Threads   |  |
|  +------------------+  |  |  +------------------+  |
+------------------------+  +------------------------+
```

### Driver vs Executor Code Execution:
- Code written outside transformations executes **once on the Driver**.
- Code inside `map`, `filter`, `foreachPartition` executes **concurrently inside Executor JVM task threads**.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_Architecture_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Investigating the Driver & SparkContext Configuration

Let us inspect the live properties configured for this cluster application.

In [ ]:
conf = spark.sparkContext.getConf()
print("=== Key Cluster Configurations ===")
for key in [
    "spark.master",
    "spark.app.name",
    "spark.driver.host",
    "spark.driver.port",
    "spark.executor.memory",
    "spark.sql.shuffle.partitions",
    "spark.eventLog.enabled"
]:
    print(f"  {key:30}: {conf.get(key, 'NOT SET')}")

## 4. Tracing Driver vs Executor Execution Contexts

Watch what prints on the Driver vs what prints in Executor worker logs!

In [ ]:
import os

print(f" DRIVER is running on container: {os.uname().nodename}")

def log_worker_info(partition_index, iterator):
    worker_host = os.uname().nodename
    yield f"Task for partition {partition_index} executed on executor container: {worker_host}"

results = (
    spark.sparkContext
    .parallelize(range(10), numSlices=4)
    .mapPartitionsWithIndex(log_worker_info)
    .collect()
)

print("\n EXECUTOR execution locations:")
for r in results:
    print(f"  - {r}")

---

## Experiment: Non-Serializable Object Driver-to-Executor Passing

- **1. What are we testing?**: Observe what happens when passing an un-serializable Python object into an executor task.
- **2. What do we expect?**: Spark raises a PickleError / Py4J serialization error because task closures must be serialized over the network.

In [ ]:
import socket

# socket objects cannot be pickled or serialized across network
sock = socket.socket()

try:
    spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: (x, str(sock))).collect()
    print("Execution succeeded.")
except Exception as e:
    print(f"Caught expected serialization error: {type(e).__name__}")
finally:
    sock.close()

**3. What happened?**: Spark threw an exception during task closure serialization.  
**4. Why did it happen?**: Every variable referenced inside a transformation lambda must be serialized via pickle to executor processes.  
**5. Practical Engineering Lesson**: Instantiate connection objects (DB connections, sockets, files) INSIDE `mapPartitions` on the executor, never on the driver.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Capacity planning for a shared cluster

- **Industry**: Enterprise data platform
- **Scenario**: Several teams share a Standalone or YARN cluster and need to understand why jobs queue.
- **Problem**: A job may be WAITING because cores are allocated to another application, not because the code is slow.
- **Why this concept is relevant**: The architecture separates resource allocation (Master/YARN) from task scheduling (Driver), so the fix depends on which layer is saturated.
- **Implementation idea**: Inspect the Master UI for cores used/free and the driver UI for executor count and task concurrency.
- **Expected result**: A clear diagnosis: wait for resources, request more capacity, or reduce executor size.
- **Alternative**: Kubernetes resource quotas and namespaces.
- **Trade-offs**: The lab Master is simple but not highly available; Kubernetes adds isolation and scheduling policy.

### Use Case 2: Executor registration failure after a network change

- **Industry**: Platform engineering
- **Scenario**: After a Docker network change, executors start but never register with the driver.
- **Problem**: The driver advertises an unreachable hostname, so executor RPCs go to the wrong address.
- **Why this concept is relevant**: The driver must bind to `0.0.0.0` and advertise a hostname reachable from workers (`jupyter` in this lab).
- **Implementation idea**: Check worker logs for connection errors, verify `spark.driver.host`, and test name resolution from a worker container.
- **Expected result**: Executors register and tasks run.
- **Alternative**: A Kubernetes headless service with stable DNS.
- **Trade-offs**: Docker bridge networking is convenient locally; Kubernetes DNS is more robust for production.

---

## Business Value

- Correct architecture diagnosis avoids wasted spend on unnecessary workers or driver memory.
- Clear separation of control and data planes speeds up incident response.
- Understanding locality and allocation improves SLA predictability.
- Supports safer multi-tenant capacity planning.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Monitor Master UI capacity, driver UI executor count, and worker logs together.
- Never put the cluster manager in the data path; shuffle traffic is executor-to-executor.
- Use stable hostnames and explicit bind addresses in containerized deployments.
- Plan for driver failure separately from executor failure; a driver failure ends the application.
- In production, use a highly available cluster manager and a monitored driver host.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Map the control plane and data plane

**Steps**

1. Open the Master UI and record cores used/free.
2. Open the driver UI Executors tab and record executor count and task slots.
3. Run a `groupBy` and watch the Stages tab for shuffle read/write.
4. Compare the Master UI during the job with the driver UI.

**What to observe**: The Master shows resource allocation; the driver UI shows task and shuffle activity.

**Healthy result**: You can point to the exact component responsible for allocation versus execution.

**Problematic result**: A job waiting for cores shows no active tasks even though the code is valid.

**Engineering takeaway**: Diagnose the layer first: allocation, registration, scheduling, or data processing.

---

## Trade-offs and Alternatives

### Advantages

- Clear separation of concerns
- Scales task execution horizontally
- Locality-aware scheduling

### Limitations

- Driver is a single point of failure for the application
- Standalone Master is not highly available
- Networking is a common failure source

### When to use it

- Understanding any Spark deployment
- Debugging resource waits
- Designing container networking

### When not to use it

- Assuming the Master executes tasks
- Treating workers and executors as the same thing

### Alternatives inside Spark

- Standalone, YARN, Kubernetes cluster managers
- Dynamic allocation

### Alternatives outside Spark

- Kubernetes scheduler
- YARN Capacity Scheduler
- Cloud managed Spark control planes

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Active Application ID and Web UI URL
**Problem Statement**: Retrieve the applicationId and webUrl from sparkContext and print them.  
**Input / Context**: spark.sparkContext

In [ ]:
# Exercise 1: App ID and UI URL

### Exercise 2 (Intermediate): Count Cores Used Across Worker Nodes
**Problem Statement**: Run a job with 8 partitions and verify how many unique container hostnames contributed.  
**Input / Context**: sc.parallelize(range(80), 8)

In [ ]:
# Exercise 2: Count active worker nodes

### Exercise 3 (Advanced): Demonstrate mapPartitions DB Connection Pattern
**Problem Statement**: Use mapPartitions to simulate opening a database connection once per partition rather than once per row.  
**Input / Context**: RDD of 100 elements across 4 partitions

In [ ]:
# Exercise 3: mapPartitions connection pattern

---

## Production Debugging Exercise

### Executors never register

**Scenario**: The Master shows executors launched, but the driver UI shows zero executors and tasks never start.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check worker logs for `Connection refused` or unknown host errors.
2. Verify `spark.driver.host` and `spark.driver.bindAddress`.
3. Run `getent hosts jupyter` from a worker container.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Active Application ID and Web UI URL):
```python
print("Application ID:", spark.sparkContext.applicationId)
print("Driver Web UI:", spark.sparkContext.uiWebUrl)
```

#### Solution 2 (Count Cores Used Across Worker Nodes):
```python
nodes = spark.sparkContext.parallelize(range(80), 8).map(lambda _: os.uname().nodename).distinct().collect()
print(f"Total distinct nodes executing work: {len(nodes)} -> {nodes}")
```

#### Solution 3 (Demonstrate mapPartitions DB Connection Pattern):
```python
def process_partition_with_mock_conn(iterator):
    # Opened once per partition on the executor JVM/Python worker
    conn = f"MockDBConn_{os.uname().nodename}"
    processed = []
    for item in iterator:
        processed.append(f"{item}_saved_via_{conn}")
    return iter(processed)

res = spark.sparkContext.parallelize(range(8), 2).mapPartitions(process_partition_with_mock_conn).collect()
print("Sample results from mapPartitions:", res[:4])
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Driver Role**: Coordinates DAG, breaks jobs into stages, dispatches tasks, hosts Web UI.
- **Executor Role**: Executes task threads, caches data blocks, serves shuffle data.
- **Serialization**: All code referenced in transformations must be serializable.

### Essential Technical Terms
- `Driver`: The orchestrating JVM process running the Spark application.
- `Executor`: Worker JVM process running task slots.
- `Closure`: Variables and functions packaged to be sent over network to executors.

### Common Engineering Mistakes
- **Initializing heavy resources on Driver**: Opening DB connections on driver and passing to lambdas fails with serialization errors.
- **Calling collect() on large outputs**: Causes Driver OOM.

### Interview Questions on This Topic
- **Q: What is the difference between a Worker and an Executor?**
  - *A*: A Worker is a long-lived machine daemon; an Executor is an application-specific JVM spawned by the worker.
- **Q: Where does the Spark UI run?**
  - *A*: Embedded inside the Driver process on port 4040.




### More Interview Questions

- **Q: What is the difference between a Worker and an Executor?**
  - *A*: A Worker is a long-lived node daemon that reports capacity and launches executors. An Executor is a per-application JVM that runs tasks and stores blocks.
- **Q: Does the Master participate in shuffle data transfer?**
  - *A*: No. The Master allocates resources; shuffle blocks move directly between executors, with the driver tracking metadata.

### What to Study Next
Open 01_spark_fundamentals/03_spark_session_and_spark_context.ipynb to master session management.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')